# EVASPA processing

In [ ]:
from __future__ import annotations

In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import os
from pprint import pprint

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from matplotlib.colors import ListedColormap

In [ ]:
from sweat.api import read_input_data, run_evaspa
from sweat.evaspa.config import check_config_evaspa
from sweat.evaspa.ef import EFModel

In [ ]:
import logging

from sweat.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

In [ ]:
def plot_rgb(data: xr.Dataset, bands: list[str] | None = None):
    if bands is None:
        bands = ["red", "green", "blue"]
    rgb = xr.DataArray(
        np.dstack(
            (data[bands[0]].data, data[bands[1]].data, data[bands[2]].data)
        ),
        data.coords.assign(band=["r", "g", "b"]),
        dict(data.sizes, band=3),
    )
    rgb.plot.imshow(  # type: ignore
        rgb="band",
        vmin=rgb.quantile(0.01),
        vmax=rgb.quantile(0.99),
    )
    plt.title("RGB")


def plot_band(data: xr.Dataset, band: str, percentile: float = 0.0):
    data[band].plot.imshow(  # type: ignore
        vmin=data[band].quantile(percentile),
        vmax=data[band].quantile(1 - percentile),
    )


def plot_mask(data: xr.Dataset, mask: str, invert=False):
    cmap = [(0.0, 0.0, 0.0, 0.0)]
    if len(np.unique(data[mask].data)) > 1:
        cmap = [(1.0, 0.0, 0.0, 1.0), (0.0, 0.0, 0.0, 0.0)]
    if invert:
        cmap = cmap[::-1]
    valid_cmap = ListedColormap(cmap)
    data[mask].plot(cmap=valid_cmap, add_colorbar=False)


def plot_flags(data: xr.Dataset, flags: str):
    da = data[flags]
    unique_combinations = np.unique(da.values)
    # Use a qualitative colormap with enough colors
    # (e.g., tab20 or a custom one)
    cmap = plt.get_cmap("tab20", len(unique_combinations))
    # Map each unique bitmask value to an integer index (for coloring)
    value_to_index = {v: i for i, v in enumerate(unique_combinations)}
    indexed_data = da.copy(data=np.vectorize(value_to_index.get)(da.values))
    indexed_data.plot(cmap=cmap, add_colorbar=False)
    legend_patches = [
        mpatches.Patch(color=cmap(i), label=format(v, "08b"))
        for i, v in enumerate(unique_combinations)
    ]
    plt.legend(
        handles=legend_patches,
        title="Classes present (bits)",
        bbox_to_anchor=(1.05, 1),
        loc="upper left",
        borderaxespad=0.0,
        fontsize="small",
    )


def plot_models(
    data: xr.Dataset,
    models: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(models.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over models
    for model_name in models.data_vars:
        if model_name not in ["valid", "flags"]:
            model = EFModel.create(models[model_name].attrs)
            var_name = model.var
            var = data[model.var].data
            lst = data["lst"].data
            ax = plt.subplot2grid((row, col), (i, j))
            # plot LST points
            ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)
            # edges coordinates
            var_max = np.round(np.ceil(np.nanmax(var) * 10) / 10, decimals=1)
            var_min = np.round(np.floor(np.nanmin(var) * 10) / 10, decimals=1)
            var_arr = np.linspace(var_min, var_max, num=100)
            tw_arr = model.twet(var_arr)
            td_arr = model.tdry(var_arr)
            ax.plot(
                var_arr,
                td_arr,
                color="tab:red",
                linewidth=3,
                label="dry edge",
                zorder=1,
            )
            ax.plot(
                var_arr,
                tw_arr,
                color="tab:blue",
                linewidth=3,
                label="wet edge",
                zorder=1,
            )
            # Labels
            ax.set_xlabel(var_name)
            ax.set_ylabel("Land Surface Temperature K")
            ax.set_title(f"Model {model_name}", fontsize=12)
            j += 1
            if j == 2:
                j = 0
                i += 1

    # title
    fig.suptitle("Evaporative fraction methods", fontsize=12)

    plt.show()


def plot_data(
    data: xr.Dataset,
    var_name: str,
):
    """
    Plot
    """
    # dimensions
    # gridspec inside gridspec
    _ = plt.figure(constrained_layout=True, figsize=(4, 3))

    ax = plt.gca()

    lst = data["lst"].data
    var = data[var_name].data
    # plot all points
    ax.scatter(var, lst, s=20, alpha=1, color="moccasin", clip_on=False)

    # Labels
    ax.set_xlabel(var_name)
    ax.set_ylabel("Land Surface Temperature K")

    plt.show()


def plot_dataset(
    data: xr.Dataset,
):
    """
    Plot EF models
    """
    # Set figure subplots
    nb = len(data.data_vars)
    row = int(np.ceil(nb / 2))
    col = 2
    fig = plt.figure(figsize=(4 * col, 3 * row), constrained_layout=True)

    i = 0
    j = 0
    # Loop over variables
    for var in data.data_vars:
        ax = plt.subplot2grid((row, col), (i, j))
        if var == "valid":
            valid_cmap = ListedColormap(
                [(1.0, 0.0, 0.0, 1.0), (0.0, 1.0, 0.0, 1.0)]
            )
            data[var].plot(ax=ax, cmap=valid_cmap, add_colorbar=False)
        elif var == "flags":
            da = data[var]
            unique_combinations = np.unique(da.values)
            # Use a qualitative colormap with enough colors
            # (e.g., tab20 or a custom one)
            cmap = plt.get_cmap("tab20", len(unique_combinations))
            # Map each unique bitmask value to an integer index (for coloring)
            value_to_index = {v: i for i, v in enumerate(unique_combinations)}
            indexed_data = da.copy(
                data=np.vectorize(value_to_index.get)(da.values)
            )
            indexed_data.plot(ax=ax, cmap=cmap, add_colorbar=False)
        else:
            data[var].plot(ax=ax)
        j += 1
        if j == 2:
            j = 0
            i += 1
    # title
    fig.suptitle("Dataset", fontsize=12)

    plt.show()


def plot_dem(xrds_dem: xr.Dataset):
    fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(18, 4))
    xrds_dem["height"].plot(
        ax=axes[0],
        vmin=xrds_dem["height"].min(),
        vmax=xrds_dem["height"].max(),
        cmap="RdYlGn_r",
    )
    xrds_dem["slope"].plot(
        ax=axes[1],
        vmin=xrds_dem["slope"].min(),
        vmax=xrds_dem["slope"].max(),
        cmap="Reds",
    )
    xrds_dem["aspect"].plot(
        ax=axes[2],
        vmin=xrds_dem["aspect"].min(),
        vmax=xrds_dem["aspect"].max(),
        cmap="twilight_shifted",
    )


def locate_test_data() -> str:
    """
    Locate test data directory
    """
    current = os.getcwd()
    parent = os.path.dirname(current)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data")
    parent = os.path.dirname(parent)
    if os.path.isdir(os.path.join(parent, "tests")):
        return os.path.join(parent, "tests", "data")
    msg = "Test data directory not found"
    raise FileNotFoundError(msg)

## Case: Simple execution

Input configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(locate_test_data(), "modis_test_geo.tif"),
        "date": "2018-05-16T10:00:00-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "ef": {
            "models": "default_evaspa",
        },
    },
}

#### Execution

Check input configuration

In [ ]:
config = check_config_evaspa(config_dict)

In [ ]:
pprint(config, sort_dicts=False)

Read

In [ ]:
input_data = read_input_data(config_dict["input"])

Run 

In [ ]:
inst_xr, daily_xr = run_evaspa(input_data, config["params"])

In [ ]:
inst_xr

In [ ]:
daily_xr

Visualize results

In [ ]:
plot_dataset(inst_xr)

In [ ]:
plot_flags(inst_xr, "flags")

In [ ]:
plot_dataset(daily_xr)

## Case: Detailed configuration

#### Configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(locate_test_data(), "modis_test_dem.tif"),
        "date": "2018-05-16T10:00:00-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "filtering": {},
        "ef": {
            "check": {"threshold": 0.02},
            "models": [
                {
                    "name": "model1",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "var": "albedo",
                },
                {
                    "name": "model2",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 5,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatEdge",
                    },
                    "var": "albedo",
                },
            ],
            "options": {"selection": False, "merging": "mean"},
        },
        "seb": {"use_topo": True, "models": ["kustas"], "merging": "mean"},
        "daily": {"use_topo": True, "method": "toa"},
    },
}

In [ ]:
pprint(config_dict, sort_dicts=False)

#### Execution

Check input file

In [ ]:
config = check_config_evaspa(config_dict)

Read

In [ ]:
input_data = read_input_data(config_dict["input"])

Run

In [ ]:
inst_xr, daily_xr = run_evaspa(input_data, config["params"])

Visualize results

In [ ]:
plot_dataset(daily_xr)

## Case: detailed pipeline

Additional imports

In [ ]:
from sweat.common import daily, filter
from sweat.evaspa import ef, seb
from sweat.evaspa.config import EVASPAParamsConfig

Configuration

In [ ]:
config_dict = {
    "input": {
        "path": os.path.join(locate_test_data(), "modis_test_dem.tif"),
        "date": "2018-05-16T10:00:00-00:00",
    },
    "output": {"path": "out"},
    "params": {
        "filtering": {},
        "ef": {
            "check": {"threshold": 0.02},
            "models": [
                {
                    "name": "model1",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 2,
                            "selection": "median",
                        },
                    },
                    "var": "albedo",
                },
                {
                    "name": "model2",
                    "dry_edge": {
                        "type": "LinearEdge",
                        "config": {
                            "interval_type": "density",
                            "interval_nb": 20,
                            "percentile": 5,
                            "selection": "median",
                        },
                    },
                    "wet_edge": {
                        "type": "FlatEdge",
                    },
                    "var": "albedo",
                },
            ],
            "options": {"selection": False, "merging": "mean"},
        },
        "seb": {"use_topo": True, "models": ["kustas"], "merging": "mean"},
        "daily": {"use_topo": True, "method": "toa"},
    },
}

Check configuration

In [ ]:
config = check_config_evaspa(config_dict)
# Validate parameters config
params_config = EVASPAParamsConfig.model_validate(config["params"])

Read input data

In [ ]:
data = read_input_data(config["input"])

In [ ]:
data.data_vars

In [ ]:
data

In [ ]:
plot_dataset(data[["lst", "emis", "albedo", "rsd", "rld"]])

In [ ]:
plot_dem(data[["height", "slope", "aspect"]])

Initialize EF models

In [ ]:
models, options = ef.initialize(params_config.ef.model_dump())
variables = ef.get_variables_from_models(models=models)

Filter data

In [ ]:
valid_mask, flags_mask = filter.find_valid_pixels(
    data,
    nan_config=variables,
    valid_config=params_config.filtering.model_dump(),
)
data["valid"] = valid_mask
data["flags"] = flags_mask

In [ ]:
if (data["valid"].data == 0).sum() == 0:
    print("No invalid pixels")
else:
    plot_mask(data=data, mask="valid", invert=True)

Check variability

In [ ]:
ef.check_variability(
    data["lst"], mask=data["valid"], **params_config.ef.check.model_dump()
)

Compute EF

In [ ]:
ef_xr, inst_xr = ef.run(models, data, **options)

In [ ]:
ef_xr

In [ ]:
plot_models(data, ef_xr)

Compute LE

In [ ]:
le_xr, merged_xr = seb.run(data, ef_xr, **params_config.seb.model_dump())
inst_xr = merged_xr.merge(
    inst_xr, join="override", combine_attrs="no_conflicts"
)

In [ ]:
le_xr

In [ ]:
plot_dataset(le_xr)

In [ ]:
plot_flags(le_xr, flags="flags")

In [ ]:
inst_xr

In [ ]:
plot_dataset(inst_xr)

Extrapolate at daily scale

In [ ]:
daily_xr = daily.extrapolate_at_daily_scale(
    inst_xr,
    variables=["le", "et"],
    dem=data[["height", "slope", "aspect"]],
    **params_config.daily.model_dump(),
)

In [ ]:
plot_dataset(daily_xr)